# 09 · Anomalía de precipitación con CHIRPS

**Objetivo:** Comparar precipitación reciente con una climatología histórica.

**Datos:** UCSB-CHG/CHIRPS/DAILY.

**Relevancia para política ambiental y social:** Apoya vigilancia de sequía, agricultura y seguridad alimentaria.

**Limitaciones:** La resolución es más gruesa que Sentinel-2 y no representa microclimas.


In [ ]:
# Instalar dependencias en Google Colab
!pip -q install earthengine-api geemap

import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project="TU_PROYECTO_GEE")

# Área de estudio de ejemplo: entorno de Chachapoyas, Amazonas, Perú
# Reemplázala por un polígono, activo de Earth Engine o coordenadas propias.
aoi = ee.Geometry.Point([-77.87, -6.23]).buffer(30000)

Map = geemap.Map()
Map.centerObject(aoi, 9)


In [ ]:
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").filterBounds(aoi)
recent = chirps.filterDate("2025-01-01","2025-12-31").sum()
baseline = ee.ImageCollection(
    ee.List.sequence(2001,2020).map(
        lambda y: chirps.filterDate(ee.Date.fromYMD(y,1,1), ee.Date.fromYMD(y,12,31)).sum()
    )
).mean()
anomaly = recent.subtract(baseline).clip(aoi)

Map.addLayer(anomaly, {"min":-500,"max":500,"palette":["red","white","blue"]}, "Anomalía mm")
Map
